In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('cars.csv')

In [3]:
df

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000
...,...,...,...,...,...
8123,Hyundai,110000,Petrol,First Owner,320000
8124,Hyundai,119000,Diesel,Fourth & Above Owner,135000
8125,Maruti,120000,Diesel,First Owner,382000
8126,Tata,25000,Diesel,First Owner,290000


<p>Here the column "brand" has too many brands and some brands have way too less cars, so we will perform ohe on km_driven and fuel first and then later we will perform ohe with top categories technique on "brand"</p>

<h1>1. One-Hot-Encoding using Pandas</h1>

In [4]:
pd.get_dummies(df,columns=['fuel','owner'],drop_first=True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


<h3>The "drop_first=True" removes the dummy variable trap by removing one of the OHE columns from every case. Here CNG and First Owner is removed.</h3>

<h1>2. One-Hot-Encoding using Sklearn</h1>

In [5]:
from sklearn.model_selection import train_test_split
x=df.drop(['selling_price'],axis=1)
y=df['selling_price']
x_train,x_test,y_train,y_test=train_test_split(x,
                                               y,
                                               test_size=0.2,
                                               random_state=2)

In [6]:
x_train.head()

,brand,km_driven,fuel,owner
5571,Hyundai,35000,Diesel,First Owner
2038,Jeep,60000,Diesel,First Owner
2957,Hyundai,25000,Petrol,First Owner
7618,Mahindra,130000,Diesel,Second Owner
6684,Hyundai,155000,Diesel,First Owner


In [18]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(drop='first',sparse_output=False)

<h3>As here we are performing ohe only on "fuel" and "owner" we have to separate them from x_train and x_test and perform ohe. After that we have to append "brand" and"km_driven" from x_train with these.</h3>

In [19]:
x_train_new=ohe.fit_transform(x_train[['fuel','owner']])
x_test_new=ohe.transform(x_test[['fuel','owner']])

<h3>This step will be optimized after learning column transformer.</h3>

In [20]:
x_train_new

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 1., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(6502, 7))

In [22]:
np.hstack((x_train[['brand','km_driven']].values,x_train_new))

array([['Hyundai', 35000, 1.0, ..., 0.0, 0.0, 0.0],
       ['Jeep', 60000, 1.0, ..., 0.0, 0.0, 0.0],
       ['Hyundai', 25000, 0.0, ..., 0.0, 0.0, 0.0],
       ...,
       ['Tata', 15000, 0.0, ..., 0.0, 0.0, 0.0],
       ['Maruti', 32500, 1.0, ..., 1.0, 0.0, 0.0],
       ['Isuzu', 121000, 1.0, ..., 0.0, 0.0, 0.0]],
      shape=(6502, 9), dtype=object)

<h1>3. OHE using most frequent variables.</h1>

In [32]:
counts=df['brand'].value_counts()
df['brand'].nunique()

32

In [33]:
threshold=100

In [34]:
repl=counts[counts<=threshold].index

In [42]:
pd.get_dummies(df['brand'].replace(repl,'uncommon')).sample(5)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
371,False,False,False,False,False,False,False,False,False,False,False,False,True
5986,False,False,False,False,False,False,True,False,False,False,False,False,False
793,False,False,False,False,False,True,False,False,False,False,False,False,False
7612,False,False,False,False,False,False,True,False,False,False,False,False,False
4565,False,False,False,False,False,False,False,False,False,False,True,False,False


<p>First, the frequency of every brand was counted using value_counts(), and the total number of unique brands was checked using nunique(). Then a threshold value was set to identify rare brands. All brands whose frequency was less than or equal to the threshold were selected and stored in repl using .index.
After that, all these rare brands were replaced with a common label called 'uncommon' using replace(), which reduced the number of categories and dimensionality. Finally, pd.get_dummies() performed One Hot Encoding by converting each remaining category into separate binary columns containing True/False (use dtype=int for 1/0 values), and sample(5) displayed 5 random encoded rows from the final dataframe.</p>